In [1]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

In [2]:
df = pd.read_csv("../../data/raw/01-itviec_raw.csv")

In [3]:
# Lưu số lương dòng của mỗi file để so sánh
old_len = len(df)

# Thống kê số dòng của mỗi file
print(f"Số dòng của itviec_merged.csv: {old_len}")

# --- PHẦN 1: THỐNG KÊ NULL BAN ĐẦU ---
print("\n--- THỐNG KÊ NULL:  ---")
null_counts = df.isnull().sum()
null_pct = (null_counts / old_len) * 100
missing_df = pd.DataFrame({
    'Số lượng Null': null_counts,
    '% Null': null_pct.map('{:.2f}%'.format)
})
display(missing_df)

# --- PHẦN 2: XÓA DÒNG CÓ JOB_TITLE = NULL ---
df = df.dropna(subset=['job_title']).copy()

print("\n" + "-"*50)
print(f"KẾT QUẢ SAU KHI XÓA JOB_TITLE TRỐNG:")
print(f"- Merged Dataset: Đã xóa {old_len - len(df)} dòng. Còn lại {len(df)} dòng.")

Số dòng của itviec_merged.csv: 822

--- THỐNG KÊ NULL:  ---


,Số lượng Null,% Null
url,0,0.00%
job_title,71,8.64%
company,71,8.64%
location,75,9.12%
salary,0,0.00%
experience,822,100.00%
education,822,100.00%
level,822,100.00%
employment_type,822,100.00%
industry,85,10.34%



--------------------------------------------------
KẾT QUẢ SAU KHI XÓA JOB_TITLE TRỐNG:
- Merged Dataset: Đã xóa 71 dòng. Còn lại 751 dòng.


**Nhận xét:** 

• Một số dòng có `url` nhưng `job_title` và `company` vẫn NULL do cào trúng trang trung gian thay vì trang tuyển dụng cụ thể, nên các dòng này cũng được loại bỏ.

• Các cột như `experience`, `education`, `level`, `employment_type`, `preferred_skills`, `specialization` có tỉ lệ NULL 100% do itviec.com không cung cấp dữ liệu này. Tuy nhiên, các cột vẫn được giữ lại để đồng nhất cấu trúc dữ liệu với TopCV.

In [4]:
# --- THỐNG KÊ CÁC GIÁ TRỊ LƯƠNG KHÔNG CHỨA SỐ (GIÁ TRỊ NHIỄU) ---

# Hàm để lọc các giá trị không chứa chữ số
def get_non_numeric_salaries(df):
    mask_no_digits = df['salary'].str.contains(r'\d', na=True) == False
    return df[mask_no_digits]['salary']

# Thống kê cho Person 2
print(f"\n{'-'*20} CÁC LOẠI LƯƠNG 'CHỮ' {'-'*20}")
non_num = get_non_numeric_salaries(df)
salary_counts = non_num.value_counts(dropna=False)
salary_pct = (non_num.value_counts(dropna=False, normalize=True) * 100)

stats_s2 = pd.DataFrame({
    'Số lượng': salary_counts,
    'Tần suất (%)': salary_pct.map('{:.2f}%'.format)
})
display(stats_s2)


-------------------- CÁC LOẠI LƯƠNG 'CHỮ' --------------------


,Số lượng,Tần suất (%)
salary,,
You'll love it,521,92.05%
Sign in to view salary,14,2.47%
Very attractive!!!,8,1.41%
Negotiable,4,0.71%
Competitive,4,0.71%
Thương lượng,3,0.53%
Attractive,2,0.35%
Negotiation,2,0.35%
Highly competitive,2,0.35%


In [5]:
# --- PHẦN 3: CHUẨN HÓA CỘT SALARY ---
# Biểu thức chính quy để tìm các dòng nhiễu
noise_pattern = (
    r'love it|sign in| negotiation|attractive|competitive|negotiable|'
    r'let\'s discuss|best in the market|open to negotiation|'
    r'thương lượng|thỏa thuận| thuận'
)

df['salary'] = df['salary'].fillna('Thỏa thuận')
mask = df['salary'].str.lower().str.contains(noise_pattern, na=False)
df.loc[mask, 'salary'] = 'Thỏa thuận'

print("\nĐã chuẩn hóa các dòng lương nhiễu.")

count_tt = len(df[df['salary'] == 'Thỏa thuận'])
print (f"\nSố dòng lương 'Thỏa thuận' sau khi chuẩn hóa: {count_tt} / {len(df)} dòng")
print (f"Chiếm {count_tt/len(df)*100:.2f}%" )
print(f" 5 DÒNG ĐẦU")
display(df.head())


Đã chuẩn hóa các dòng lương nhiễu.

Số dòng lương 'Thỏa thuận' sau khi chuẩn hóa: 566 / 751 dòng
Chiếm 75.37%
 5 DÒNG ĐẦU


,url,job_title,company,location,salary,experience,education,level,employment_type,industry,required_skills,preferred_skills,specialization,job_description,requirement
0,https://itviec.com/it-jobs/02-back-end-develop...,"02 Back-end Developer (Golang, PostgreSQL, AWS)",CÔNG TY TNHH AMPACS INTERNATIONAL,473 Dien Bien Phu street (194 Golden building)...,"1,000 - 2,000 USD",NaN,NaN,NaN,NaN,Manufacturing and Engineering,"Golang, PostgreSql, AWS",NaN,NaN,We are looking for a talented Golang developer...,Responsibilities:\nDevelop and maintain micros...
1,https://itviec.com/it-jobs/02-mobile-app-devel...,02 Mobile App Developer (Flutter),CÔNG TY TNHH AMPACS INTERNATIONAL,473 Dien Bien Phu street (194 Golden building)...,"1,000 - 1,500 USD",NaN,NaN,NaN,NaN,Manufacturing and Engineering,"Flutter, Swift, Kotlin, Java",NaN,NaN,We are building a modern IoT and IP camera pla...,Responsibilities\n* Develop and maintain mobil...
2,https://itviec.com/it-jobs/03-flutter-mobile-a...,"03 Flutter Mobile Apps Developer (Dart, Androi...",SHS | Chứng khoán Sài Gòn - Hà Nội,"Số 43 Phố Lý Thường Kiệt, Cua Nam, Ha Noi",15 - 50m,NaN,NaN,NaN,NaN,Financial Services,"Flutter, Android, iOS, Dart, Mobile Apps",NaN,NaN,"Thành thạo Flutter (Dart), có kinh nghiệm phát...","Thành thạo\nFlutter (Dart)\n, có kinh nghiệm p..."
3,https://itviec.com/it-jobs/03-java-backend-dev...,"03 Java Backend Developer (Spring, SQL)",SHS | Chứng khoán Sài Gòn - Hà Nội,"Số 43 Phố Lý Thường Kiệt, Cua Nam, Ha Noi",15 - 50m,NaN,NaN,NaN,NaN,Financial Services,"Java, Database, SQL, Oracle, Microservices, Sp...",NaN,NaN,Phát triển và tối ưu các API/dịch vụ back-end ...,Có từ\n02 năm kinh nghiệm\ntrở lên\nThành thạo...
4,https://itviec.com/it-jobs/03-java-kotlin-deve...,"03 Java/Kotlin Developers (All level, English)",Topicus Vietnam,"106 Nguyen Van Troi Street, Phu Nhuan, Ho Chi ...",Thỏa thuận,NaN,NaN,NaN,NaN,Software Products and Web Services,"Java, AWS, PostgreSql, Spring Boot, English, K...",NaN,NaN,"About Us\nAt Topicus, we believe IT should ser...",1-5+ years of experience in Kotlin or Java usi...


## **Tiếp theo ta sẽ tiến hành thống nhất các nội dung ở cột salary về 1 chuẩn (đơn vị triệu đồng)**

In [6]:
def standardize_itviec_salary(salary_str):
    if pd.isna(salary_str) or salary_str == 'Thỏa thuận':
        return 0, 0, 0

    s = str(salary_str).lower().strip()
    
    # 1. Xử lý các dấu gạch ngang khác nhau (– vs -) và khoảng cách
    s = s.replace('–', '-').replace(',', '') # Xóa dấu phẩy phân cách hàng nghìn 
    
    # 2. Kiểm tra đơn vị ngoại tệ
    is_usd = any(kw in s for kw in ['usd', '$'])
    
    # 3. Bóc tách tất cả các số (bao gồm cả số nguyên và số thập phân)
    # Ví dụ: "1.5" hoặc "1200" hoặc "15.000.000"
    # Lưu ý: Vì đã xóa dấu phẩy ở bước 1, nên dấu chấm ở đây có thể là phân cách hàng nghìn (15.000.000)
    # Ta xóa dấu chấm nếu nó nằm giữa các cụm số để đưa về số nguyên thuần túy
    if '.' in s:
        # Nếu dấu chấm phân tách hàng nghìn (ví dụ 10.000.000) thì xóa
        # Nếu là số thập phân (ví dụ 1.5 triệu) thì giữ
        parts = s.split('.')
        if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) > 2): 
            s = s.replace('.', '')

    numbers = re.findall(r'\d+', s)
    numbers = [float(n) for n in numbers]

    if not numbers:
        return 0, 0, 0

    min_val, max_val = 0, 0

    # 4. Xác định Logic Min/Max
    if 'up to' in s or 'upto' in s:
        min_val, max_val = 0, numbers[0]
    elif 'từ' in s or 'from' in s:
        min_val, max_val = numbers[0], 0
    elif '-' in s and len(numbers) >= 2:
        min_val, max_val = numbers[0], numbers[1]
    else:
        min_val = max_val = numbers[0]

    # 5. Chuẩn hóa đơn vị về "Triệu VNĐ"
    def to_million(val):
        if val <= 0: return 0
        # Nếu là USD
        if is_usd:
            return (val * 25000) / 1000000
        # Nếu con số khổng lồ (ví dụ 70.000.000)
        if val >= 1000000:
            return val / 1000000
        # Nếu là con số nhỏ nhưng không phải USD (ví dụ 15m, 20tr, 30) -> Nó là triệu sẵn rồi
        return val

    min_salary = to_million(min_val)
    max_salary = to_million(max_val)

    # 6. Tính Average
    if min_salary > 0 and max_salary > 0:
        avg_salary = (min_salary + max_salary) / 2
    else:
        avg_salary = max(min_salary, max_salary)

    return round(min_salary, 2), round(max_salary, 2), round(avg_salary, 2)

# --- THỰC THI TRÊN DATA ---
for df in [df]:
    results = df['salary'].apply(lambda x: standardize_itviec_salary(x))
    df[['min_salary', 'max_salary', 'avg_salary']] = pd.DataFrame(results.tolist(), index=df.index)

print(" Đã xử lý xong các cases lương đặc thù của TopCV!")
# print(f" 5 DÒNG ĐẦU PERSON 1 SAU KHI CHUẨN HÓA LƯƠNG:")
display(df[['salary', 'min_salary', 'max_salary', 'avg_salary']].head(10))

 Đã xử lý xong các cases lương đặc thù của TopCV!


,salary,min_salary,max_salary,avg_salary
0,"1,000 - 2,000 USD",25.0,50.0,37.50
1,"1,000 - 1,500 USD",25.0,37.5,31.25
2,15 - 50m,15.0,50.0,32.50
3,15 - 50m,15.0,50.0,32.50
4,Thỏa thuận,0.0,0.0,0.00
5,15 - 50m,15.0,50.0,32.50
6,15 - 50m,15.0,50.0,32.50
7,Thỏa thuận,0.0,0.0,0.00
8,Thỏa thuận,0.0,0.0,0.00
9,Thỏa thuận,0.0,0.0,0.00


In [7]:
salary_col = df["salary"].astype(str).str.lower().str.strip()
df["is_null_salary"] = df["salary"].isna()
df["is_thoa_thuan"] = salary_col.str.contains("thỏa thuận",regex=True,na=False)
df["is_missing_salary"] = (
    df["is_null_salary"] |
    df["is_thoa_thuan"] |
    salary_col.isin(["", "nan", "none"])
)
print(f"Tỷ lệ dữ liệu KHÔNG có lương thực tế: {df['is_missing_salary'].mean()*100:.2f}%")
print(f"Tỷ lệ dữ liệu có lương thực tế: {(~df['is_missing_salary']).mean()*100:.2f}%")
print(f"Số dòng có lương thực tế: {(~df['is_missing_salary']).sum()} / {len(df)} dòng")
print(f"Số dòng thiếu thông tin lương: {df['is_missing_salary'].sum()} / {len(df)} dòng")
print(f"Số dòng có lương 'Thỏa thuận': {df['is_thoa_thuan'].sum()} / {len(df)} dòng")

Tỷ lệ dữ liệu KHÔNG có lương thực tế: 75.37%
Tỷ lệ dữ liệu có lương thực tế: 24.63%
Số dòng có lương thực tế: 185 / 751 dòng
Số dòng thiếu thông tin lương: 566 / 751 dòng
Số dòng có lương 'Thỏa thuận': 566 / 751 dòng


## **Chuẩn hóa cột Location: Chỉ giữ lại tên thành phố**

Ví dụ: Ho Chi Minh

In [8]:
df["location"] = df["location"].str.split(",").str[-1].str.strip()
df["location"].value_counts()

location
Ho Chi Minh                                           383
Ha Noi                                                281
Da Nang                                                34
Ho Chi Minh - Fresher Accepted                         29
Ha Noi - Fresher Accepted                              13
Dong Nai                                                2
Lao PDR                                                 1
Ho Chi Minh - 128 rue La Boetie 75008 Paris FRANCE      1
Nhật Bản                                                1
Bac Ninh                                                1
Overseas                                                1
Name: count, dtype: int64

In [9]:
df["location"] = (
    df["location"]
    .str.split(",").str[-1]          # lấy phần cuối sau dấu ,
    .str.split("-").str[0]           # bỏ phần sau dấu -
    .str.strip()
)
df["location"].value_counts()

location
Ho Chi Minh    413
Ha Noi         294
Da Nang         34
Dong Nai         2
Lao PDR          1
Nhật Bản         1
Bac Ninh         1
Overseas         1
Name: count, dtype: int64

In [10]:
df[df["location"].isnull()]

,url,job_title,company,location,salary,experience,education,level,employment_type,industry,...,preferred_skills,specialization,job_description,requirement,min_salary,max_salary,avg_salary,is_null_salary,is_thoa_thuan,is_missing_salary
28,https://itviec.com/it-jobs/ai-engineer-deep-le...,Remote AI Engineer (Deep Learning Algorithm/ P...,VisionTech Global Limited,NaN,Thỏa thuận,NaN,NaN,NaN,NaN,AI Software & Services,...,NaN,NaN,VisionTech is building an algorithm team capab...,Location: Not based in Taiwan. We are looking ...,0.0,0.0,0.00,False,True,True
184,https://itviec.com/it-jobs/data-engineer-ai-da...,Remote Data Engineer - AI Data Platform Japan ...,"SalesNow Co., Ltd.",NaN,"1,000 - 2,500 USD",NaN,NaN,NaN,NaN,AI Software & Services,...,NaN,NaN,What Makes SalesNow Different\nData scale: 8B ...,Must Have\n2+ years\nof professional data engi...,25.0,62.5,43.75,False,False,False
285,https://itviec.com/it-jobs/fullstack-software-...,Fullstack Software Engineer,Revve AI,NaN,Negotiation,NaN,NaN,NaN,NaN,Software Products and Web Services,...,NaN,NaN,"· Design, build, and deploy new features in...","· Bachelor’s degree in Computer Science, En...",0.0,0.0,0.00,False,False,False
682,https://itviec.com/it-jobs/senior-java-enginee...,"Senior Java Engineer (Spring Boot, Microservic...",EPAM Vietnam,NaN,Thỏa thuận,NaN,NaN,NaN,NaN,IT Services and IT Consulting,...,NaN,NaN,EPAM Vietnam\nis hiring a\nSenior Java Develop...,At least 5 years of experience in software dev...,0.0,0.0,0.00,False,True,True


## **Tiến hành extract dữ liệu từ job_cho các cột `level`, `experience`, `employment_type`, `education`**

**Hàm xử lý cho cột employment_type**

In [11]:
def extract_employment_type(text):
    if not isinstance(text, str): return "Toàn thời gian"
    text_lower = text.lower()
    
    # Bán thời gian
    if any(k in text_lower for k in ["part-time", "bán thời gian", "part time", "parttime"]):
        return "Bán thời gian"
    # Thời vụ
    if any(k in text_lower for k in ["contract", "freelance", "thời vụ", "hợp đồng", "hợp đồng ngắn hạn",
                                     "project based", "project-based", "consultant"]):
        return "Thời vụ"
    # Hybrid
    if re.search(r'\bhybrid\b', text_lower):
        
        # Ưu tiên các cụm rõ ràng chỉ hình thức làm việc (cao nhất)
        work_related_patterns = [
            "hybrid working", "hybrid work", "làm việc hybrid", 
            "hybrid remote", "hybrid model", "hybrid arrangement", 
            "hybrid schedule", "hybrid onsite", "days hybrid"
        ]
        if any(p in text_lower for p in work_related_patterns):
            return "Hybrid"
        # Bắt các trường hợp dính ký tự đặc biệt: Hybrid(, Hybrid-, Hybrid~, Hybrid:
        if re.search(r'\bhybrid\s*[\(\-\~\:\[]', text_lower):
            return "Hybrid"
        # Nếu không có dấu hiệu làm việc rõ ràng → kiểm tra technical terms
        tech_patterns = [
            "hybrid system", "hybrid architecture", "hybrid app", "hybrid mobile",
            "hybrid cloud", "hybrid integration", "hybrid protocol", 
            "hybrid infrastructure", "hybrid engine", "hybrid solution", 
            "hybrid platform", "hybrid framework"
        ]
        is_technical = any(tp in text_lower for tp in tech_patterns)
        # Nếu không phải technical → coi là Hybrid làm việc
        if not is_technical:
            return "Hybrid"
    # Remote/WFH
    if any(k in text_lower for k in ["remote", "làm từ xa", "wfh", "work from home", 
                                     "fully remote", "remote only", "remote position", "remote work"]):
        return "Remote"

    # Mặc định là Full-time
    return "Toàn thời gian"

**Hàm xử lý cho cột level**

In [12]:
def extract_level(text):
    if not isinstance(text, str) or not text.strip():
        return "Nhân viên"
    
    text_lower = text.lower()
    
    # Cấp quản lý cao nhất
    if any(k in text_lower for k in ["manager", "trưởng phòng", "head of", "director", 
                                     "giám đốc", "vp", "cto", "ceo", "chief"]):
        return "Quản lý / Giám sát"
    
    # Nhóm Lead / Tech Lead (ưu tiên trước Senior)
    elif any(k in text_lower for k in ["tech lead", "technical lead", "lead", 
                                       "trưởng nhóm", "team lead", "group lead"]):
        return "Trưởng nhóm"
    
    # Senior & các cấp cao khác
    elif any(k in text_lower for k in ["senior", "expert", "chuyên gia", "principal", 
                                       "staff", "architect"]):
        return "Senior"
    
    # Junior / Mid
    elif any(k in text_lower for k in ["junior", "mid-level", "middle", " mid "]):
        return "Junior"
    
    # Fresher
    elif any(k in text_lower for k in ["fresher", "mới tốt nghiệp", "entry level", "entry-level"]):
        return "Fresher"
    
    # Intern (xét cuối)
    elif any(k in text_lower for k in ["intern", "thực tập", "thực tập sinh"]):
        return "Thực tập sinh"
    
    return "Nhân viên"

**Hàm xử lý cho cột experience**

In [13]:
def extract_experience(text):
    if not isinstance(text, str) or not text.strip():
        return "Không yêu cầu"
    
    text_lower = text.lower()
    
    # Không yêu cầu kinh nghiệm
    if any(k in text_lower for k in ["fresh", "mới tốt nghiệp", "no experience", 
                                     "không yêu cầu", "không cần kinh nghiệm", "0 năm"]):
        return "Dưới 1 năm"

    # Kiểm tra tuổi - chỉ dùng cho việc lọc sau, không chặn cứng
    is_age_mention = any(phrase in text_lower for phrase in [
        "years old", "year old", "tuổi", "age", "under 30", "over 30", "below 30"
    ])

    # ==================== ƯU TIÊN 1: DẤU "+" ====================
    # Cải tiến regex: bắt được cả "2+ years", "2 + years", "2years+"
    plus_match = re.search(r'(\d+)\s*\+\s*(?:năm|year|years|yr)', text_lower)  # Dấu + đứng sau số là quan trọng nhất
    if plus_match:
        years = int(plus_match.group(1))
        if years == 0:
            return "Dưới 1 năm"
        else:
            return f"Trên {years} năm"

    # ==================== ƯU TIÊN 2: DẠNG RANGE ====================
    range_match = re.search(r'(\d+)\s*[-–~]\s*(\d+)\s*(?:năm|year|years)', text_lower)
    if range_match and not is_age_mention:
        min_y = int(range_match.group(1))
        if min_y > 20: 
            pass 
        else:
            # Nếu min là 0 thì trả về "Dưới 1 năm", còn lại trả về con số
            return "Dưới 1 năm" if min_y == 0 else f"{min_y} năm"

    # ==================== ƯU TIÊN 3: CÁC TRƯỜNG HỢP THÔNG THƯỜNG ====================
    exp_patterns = [
        r'(?:ít nhất|tối thiểu|minimum|from|at least|trên|over)\s*(\d+)\s*(?:năm|year|years)',
        r'(\d+)\s*(?:năm|year|years)\s*(?:kinh nghiệm|experience|exp)?',
        r'(?:kinh nghiệm|experience|exp)[:\s-]*(\d+)',
        r'(\d+)\s*(?:năm|year|years)',
    ]
    
    for pattern in exp_patterns:
        match = re.search(pattern, text_lower)
        if match:
            years = int(match.group(1))
            
            # Chỉ skip nếu nghi ngờ là tuổi (số lớn + có mention tuổi)
            if is_age_mention and years > 18:
                continue
                
            return "Dưới 1 năm" if years <= 1 else f"{years} năm"

    # ==================== FALLBACK ====================
    if any(k in text_lower for k in ["senior", "nhiều năm", "expert", "chuyên gia", "lead"]):
        return "Trên 3 năm"
    
    return "Không yêu cầu"

**Hàm xử lý cho cột education**

In [14]:
def extract_education(text):
    if not isinstance(text, str) or not text.strip(): 
        return "Không yêu cầu"
    text_lower = text.lower()
    education = "Không yêu cầu"
    
    edu_keywords = [
        "thạc sĩ", "master", "msc", "postgraduate", "cao học",
        "đại học", "university", "bachelor", "cử nhân", "bsc", 
        "tốt nghiệp đại học", "graduate", "degree", "tốt nghiệp",
        "cao đẳng", "college", "associate"
    ]
    
    if any(k in text_lower for k in edu_keywords):
        if any(k in text_lower for k in ["thạc sĩ", "master", "msc", "cao học"]):
            education = "Thạc sĩ trở lên"
        elif any(k in text_lower for k in ["đại học", "university", "bachelor", "cử nhân", "tốt nghiệp đại học", "graduate", "degree"]):
            education = "Đại học trở lên"
        elif any(k in text_lower for k in ["cao đẳng", "college"]):
            education = "Cao đẳng trở lên"

    # Fallback cho các job chỉ ghi "Tốt nghiệp" hoặc "Graduate"
    if education == "Không yêu cầu" and ("tốt nghiệp" in text_lower or "graduate" in text_lower):
        education = "Đại học trở lên"

    return education

**Áp dụng**

Cell này sẽ tạo ra một cột nội dung tổng hợp và áp dụng đồng thời cả 4 hàm trên.

In [15]:
cols = ['job_title', 'job_description', 'requirement']

df['level'] = df[cols].fillna('').apply(lambda x: extract_level(" ".join(x)), axis=1)
df['experience'] = df[cols].fillna('').apply(lambda x: extract_experience(" ".join(x)), axis=1)
df['employment_type'] = df[cols].fillna('').apply(lambda x: extract_employment_type(" ".join(x)), axis=1)
df['education'] = df[cols].fillna('').apply(lambda x: extract_education(" ".join(x)), axis=1)

print(" Đã trích xuất xong dữ liệu cho 4 cột mới!")
display(df[['job_title', 'level', 'experience', 'employment_type', 'education']].head(10))

 Đã trích xuất xong dữ liệu cho 4 cột mới!


,job_title,level,experience,employment_type,education
0,"02 Back-end Developer (Golang, PostgreSQL, AWS)",Senior,Trên 1 năm,Hybrid,Không yêu cầu
1,02 Mobile App Developer (Flutter),Nhân viên,Dưới 1 năm,Toàn thời gian,Không yêu cầu
2,"03 Flutter Mobile Apps Developer (Dart, Androi...",Nhân viên,Không yêu cầu,Toàn thời gian,Không yêu cầu
3,"03 Java Backend Developer (Spring, SQL)",Nhân viên,2 năm,Toàn thời gian,Không yêu cầu
4,"03 Java/Kotlin Developers (All level, English)",Senior,Trên 5 năm,Toàn thời gian,Không yêu cầu
5,"03 Tester (Manual Tester, Automation Test, QA QC)",Nhân viên,Dưới 1 năm,Toàn thời gian,Không yêu cầu
6,"03 UI-UX Desginer (Photoshop, Illustrator, Figma)",Nhân viên,2 năm,Toàn thời gian,Không yêu cầu
7,"04 Senior Fullstack Dev (Golang, Java, NodeJS,...",Trưởng nhóm,5 năm,Remote,Đại học trở lên
8,05 Senior/Lead Product Manager ( Fintech),Quản lý / Giám sát,Trên 3 năm,Toàn thời gian,Đại học trở lên
9,05 System Admin (Linux),Nhân viên,Không yêu cầu,Toàn thời gian,Đại học trở lên


**Thống kê các dòng "Không yêu cầu" ở cột `experience` và `education`**

In [16]:
exp_missing_val = "Không yêu cầu" 
edu_missing_val = "Không yêu cầu"

# Lọc các dòng bị thiếu
missing_exp = df[df['experience'] == exp_missing_val]
missing_edu = df[df['education'] == edu_missing_val]

print(f" THỐNG KÊ DỮ LIỆU KHÔNG XÁC ĐỊNH (Tổng số dòng: {len(df)})")
print("-" * 45)
print(f" Experience chưa xác định: {len(missing_exp)} dòng ({(len(missing_exp)/len(df))*100:.2f}%)")
print(f" Education chưa xác định:  {len(missing_edu)} dòng ({(len(missing_edu)/len(df))*100:.2f}%)")

 THỐNG KÊ DỮ LIỆU KHÔNG XÁC ĐỊNH (Tổng số dòng: 751)
---------------------------------------------
 Experience chưa xác định: 64 dòng (8.52%)
 Education chưa xác định:  340 dòng (45.27%)


In [17]:
# Lưu file
df.to_csv("../../data/interim/04-itviec_cleaned.csv", index=False)